### observation01

10개 smoke sample에 대한 수작업 정성 분석

In [1]:
# ps -ef | grep jupyter
# 실행중인 jupyter notebook 프로세스 확인

In [2]:
# self-plan 결과 분석
from pathlib import Path
import json

import pandas as pd


SELF_PLAN_PATH = Path(
    "../self_plan_stdin/results.jsonl"
)


def load_jsonl(path: Path) -> pd.DataFrame:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(
            file,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON at line {line_number}"
                ) from error

    return pd.DataFrame(records)


self_plan_df = load_jsonl(SELF_PLAN_PATH)

print(f"Loaded records: {len(self_plan_df)}")

Loaded records: 10


In [3]:
# 계획 단계 추출 함수
def get_strategy_step(
    row: pd.Series,
    step_name: str,
) -> dict:
    trace = row.get("strategy_trace", [])

    for step in trace:
        if step.get("name") == step_name:
            return step

    raise KeyError(
        f"Strategy step not found: {step_name}"
    )


def get_generated_plan(
    row: pd.Series,
) -> str:
    plan_step = get_strategy_step(
        row,
        "plan_generation",
    )
    return plan_step["raw_output"]

In [4]:
# 문제 하나를 분석하기 위한 출력 함수

def show_self_plan_case(
    problem_id: str,
) -> None:
    matched = self_plan_df.loc[
        self_plan_df["problem_id"] == problem_id
    ]

    if matched.empty:
        raise KeyError(
            f"Unknown problem_id: {problem_id}"
        )

    row = matched.iloc[0]
    plan = get_generated_plan(row)

    print("=" * 100)
    print(
        f"{row['problem_id']} | "
        f"{row['title']} | "
        f"{row['difficulty']} | "
        f"{row['status']}"
    )
    print("=" * 100)

    print("\n[Problem]")
    print(row["problem"])

    print("\n[Generated Plan]")
    print(plan)

    print("\n[Extracted Code]")
    print(row["extracted_code"])

    print("\n[Evaluation]")
    print(f"Passed          : {row['passed']}")
    print(f"Status          : {row['status']}")
    print(
        f"Passed tests    : "
        f"{row['passed_tests']} / "
        f"{row['total_tests']}"
    )
    print(f"Error message   : {row['error_message']}")

    failed_tests = [
        test
        for test in row["test_results"]
        if not test["passed"]
    ]

    if failed_tests:
        first_failed = failed_tests[0]

        print("\n[First Failed Test]")
        print(
            f"Test index      : "
            f"{first_failed['test_index']}"
        )
        print(
            f"Status          : "
            f"{first_failed['status']}"
        )

        print("\nInput:")
        print(first_failed["input_text"])

        print("\nExpected:")
        print(first_failed["expected_output"])

        print("\nActual:")
        print(first_failed["actual_output"])

        print("\nstderr:")
        print(first_failed["stderr"])

In [5]:
# show_self_plan_case("1873_A")

# 1873_A
# 1873_B
# 1873_D
# 1883_B
# 1883_C
# 1899_A
# 1899_B
# 1899_C
# 1899_D
# abc301_a

In [6]:
# 라벨링 테이블 생성
analysis_df = self_plan_df[
    [
        "problem_id",
        "title",
        "difficulty",
        "passed",
        "status",
        "passed_tests",
        "total_tests",
    ]
].copy()

analysis_df["plan_correctness"] = ""
analysis_df["implementation_fidelity"] = ""
analysis_df["primary_bottleneck"] = ""
analysis_df["analysis_note"] = ""

display(analysis_df)

,problem_id,title,difficulty,passed,status,passed_tests,total_tests,plan_correctness,implementation_fidelity,primary_bottleneck,analysis_note
0,1873_A,A. Short Sort,easy,False,WRONG_ANSWER,3,5,,,,
1,1873_B,B. Good Kid,easy,False,WRONG_ANSWER,0,13,,,,
2,1873_D,D. 1D Eraser,easy,False,WRONG_ANSWER,0,13,,,,
3,1883_B,B. Chemistry,medium,False,WRONG_ANSWER,0,4,,,,
4,1883_C,C. Raspberries,medium,False,WRONG_ANSWER,8,13,,,,
5,1899_A,A. Game with Integers,easy,False,WRONG_ANSWER,2,13,,,,
6,1899_B,B. 250 Thousand Tons of TNT,hard,False,RUNTIME_ERROR,0,13,,,,
7,1899_C,C. Yarik and Array,hard,False,WRONG_ANSWER,6,13,,,,
8,1899_D,D. Yarik and Musical Notes,hard,False,WRONG_ANSWER,12,13,,,,
9,abc301_a,Overall Winner,easy,False,WRONG_ANSWER,13,15,,,,


In [7]:
PLAN_CORRECTNESS_LABELS = {
    "Correct",
    "Partially Correct",
    "Incorrect",
    "Unclear",
}

FIDELITY_LABELS = {
    "High",
    "Medium",
    "Low",
    "Unclear",
}

BOTTLENECK_LABELS = {
    "None",
    "Planning",
    "Coding",
    "Both",
    "Unclear",
}

In [8]:
# 라벨 입력함수
def label_case(
    problem_id: str,
    *,
    plan_correctness: str,
    implementation_fidelity: str,
    primary_bottleneck: str,
    analysis_note: str,
) -> None:
    if (
        plan_correctness
        not in PLAN_CORRECTNESS_LABELS
    ):
        raise ValueError(
            f"Invalid plan_correctness: "
            f"{plan_correctness}"
        )

    if (
        implementation_fidelity
        not in FIDELITY_LABELS
    ):
        raise ValueError(
            f"Invalid implementation_fidelity: "
            f"{implementation_fidelity}"
        )

    if (
        primary_bottleneck
        not in BOTTLENECK_LABELS
    ):
        raise ValueError(
            f"Invalid primary_bottleneck: "
            f"{primary_bottleneck}"
        )

    mask = (
        analysis_df["problem_id"] == problem_id
    )

    if not mask.any():
        raise KeyError(
            f"Unknown problem_id: {problem_id}"
        )

    analysis_df.loc[
        mask,
        "plan_correctness",
    ] = plan_correctness

    analysis_df.loc[
        mask,
        "implementation_fidelity",
    ] = implementation_fidelity

    analysis_df.loc[
        mask,
        "primary_bottleneck",
    ] = primary_bottleneck

    analysis_df.loc[
        mask,
        "analysis_note",
    ] = analysis_note

In [9]:
# 수작업으로 진행(gpt활용)

label_case(
    "1873_A",
    plan_correctness="Incorrect",
    implementation_fidelity="High",
    primary_bottleneck="Planning",
    analysis_note=(
        "The plan incorrectly characterizes one-swap "
        "reachability using adjacent inversions and "
        "misclassifies cba. The code faithfully implements "
        "the incorrect decision rule."
    ),
)
label_case(
    "1873_B",
    plan_correctness="Partially Correct",
    implementation_fidelity="Low",
    primary_bottleneck="Coding",
    analysis_note=(
        "The plan identifies the correct brute-force idea of "
        "incrementing each digit and comparing the resulting full "
        "array product, although its edge-case statements are incorrect. "
        "The code does not implement the planned product computation; "
        "instead, it treats the digits as decimal positions using powers "
        "of ten. The primary failure is therefore in code generation."
    ),
)

label_case(
    "1873_D",
    plan_correctness="Incorrect",
    implementation_fidelity="Medium",
    primary_bottleneck="Both",
    analysis_note=(
        "The plan incorrectly models the task as a sliding-window "
        "optimization based on the number of black cells in each window. "
        "The correct solution is a left-to-right greedy scan that performs "
        "an operation at each first uncovered black cell and skips k positions. "
        "The code follows the plan's sliding-window direction but further "
        "reduces the objective to the minimum black count of any window, "
        "which does not represent the number of operations needed for the "
        "entire string."
    ),
)

label_case(
    "1883_B",
    plan_correctness="Partially Correct",
    implementation_fidelity="Medium",
    primary_bottleneck="Both",
    analysis_note=(
        "The plan correctly recognizes that palindrome rearrangement "
        "depends on the number of odd character frequencies, but it does "
        "not derive the exact condition for deleting exactly k characters. "
        "The correct criterion is odd_count <= k + 1. The implementation "
        "counts odd frequencies in the original string and simply checks "
        "odd_count <= 1, effectively ignoring the impact of k. Therefore, "
        "both incomplete planning and incorrect implementation contribute "
        "to the failure."
    ),
)

label_case(
    "1883_C",
    plan_correctness="Incorrect",
    implementation_fidelity="Low",
    primary_bottleneck="Both",
    analysis_note=(
        "The plan incorrectly formulates the problem as a dynamic "
        "program over digit increment counts and misses the key case-based "
        "solution, especially the special handling for k=4 using either "
        "one multiple of four or two even factors. The implementation does "
        "not compute the actual increment cost for any array element and "
        "instead updates the product modulo k using an unrelated loop index. "
        "Thus both the planning abstraction and the code implementation are "
        "fundamentally incorrect."
    ),
)

label_case(
    "1899_A",
    plan_correctness="Incorrect",
    implementation_fidelity="High",
    primary_bottleneck="Planning",
    analysis_note=(
        "The plan derives an incorrect and internally inconsistent "
        "winning condition. It claims that First wins when n is already "
        "divisible by 3, although the winning condition applies only after "
        "First makes a move. The correct rule is First when n % 3 != 0 "
        "and Second when n % 3 == 0. The code faithfully implements the "
        "incorrect plan and returns First for every input."
    ),
)

label_case(
    "1899_B",
    plan_correctness="Incorrect",
    implementation_fidelity="Medium",
    primary_bottleneck="Both",
    analysis_note=(
        "The plan incorrectly reformulates the problem as a greedy solution "
        "over sorted weights, whereas the original order of boxes must be "
        "preserved and block sums should be evaluated for every divisor of n. "
        "The implementation follows the incorrect sorting-based direction but "
        "also fails to compute all truck block sums, considering only the first "
        "block. In addition, the submitted code contains a Python variable scope "
        "bug (`data` becomes a local variable), causing a runtime error before "
        "the algorithm executes."
    ),
)

label_case(
    "1899_C",
    plan_correctness="Partially Correct",
    implementation_fidelity="Medium",
    primary_bottleneck="Both",
    analysis_note=(
        "The plan correctly identifies a Kadane-style solution and the need "
        "to continue or restart based on alternating parity. However, its "
        "max_odd/max_even invariant does not preserve the requirement that "
        "the subarray be contiguous and end at the immediately preceding "
        "position. The implementation follows this flawed state design and "
        "also extends an even-ending state with another even value, and "
        "similarly for odd values, violating the alternating-parity condition. "
        "Thus both incomplete planning and incorrect implementation contribute "
        "to the failure."
    ),
)

label_case(
    "1899_D",
    plan_correctness="Incorrect",
    implementation_fidelity="High",
    primary_bottleneck="Planning",
    analysis_note=(
        "The plan incorrectly concludes that the equality holds only "
        "when the two note values are equal. For b_i = 2^a_i and "
        "b_j = 2^a_j, the condition reduces to "
        "a_i * 2^a_j = a_j * 2^a_i, which holds when a_i = a_j "
        "and also for the special pair {1, 2}. The implementation "
        "faithfully counts only equal-value pairs and therefore omits "
        "all cross pairs between values 1 and 2."
    ),
)

label_case(
    "abc301_a",
    plan_correctness="Correct",
    implementation_fidelity="Medium",
    primary_bottleneck="Coding",
    analysis_note=(
        "The plan correctly states that the winner is determined by total "
        "wins, with ties broken by who reaches the final tied count first. "
        "The implementation correctly counts total wins but handles ties "
        "incorrectly by printing the winner of the first game rather than "
        "tracking when each player reaches the final tied count. The primary "
        "failure is therefore in code generation."
    ),
)

# display(
#     analysis_df.loc[
#         analysis_df["problem_id"] == "1873_A"
#     ]
# )

In [10]:
# show_self_plan_case("1873_A")

# # 1873_A
# # 1873_B
# # 1873_D
# # 1883_B
# # 1883_C
# # 1899_A
# # 1899_B
# # 1899_C
# # 1899_D
# # abc301_a

In [11]:
def get_unlabeled_problem_ids() -> list[str]:
    unlabeled = analysis_df.loc[
        analysis_df["plan_correctness"] == "",
        "problem_id",
    ]

    return unlabeled.tolist()


print(get_unlabeled_problem_ids())

[]


In [12]:
unlabeled_ids = get_unlabeled_problem_ids()

if unlabeled_ids:
    show_self_plan_case(unlabeled_ids[0])
else:
    print("All cases are labeled.")

All cases are labeled.


In [13]:
ANALYSIS_PATH = Path(
    "../self_plan_stdin/failure_analysis.csv"
)


def save_failure_analysis() -> None:
    analysis_df.to_csv(
        ANALYSIS_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"Saved analysis: "
        f"{ANALYSIS_PATH.resolve()}"
    )


save_failure_analysis()

Saved analysis: /home/dibaeck/workspace/project_sLM_planning/phase1_planning_bottleneck/outputs/self_plan_stdin/failure_analysis.csv


In [14]:
# if ANALYSIS_PATH.exists():
#     analysis_df = pd.read_csv(
#         ANALYSIS_PATH,
#         keep_default_na=False,
#     )

#     print(
#         f"Loaded previous labels: "
#         f"{len(analysis_df)}"
#     )

---
라벨링 완료 후 요약

In [15]:
def validate_failure_labels(
    analysis_df: pd.DataFrame,
) -> None:
    label_columns = [
        "plan_correctness",
        "implementation_fidelity",
        "primary_bottleneck",
    ]

    incomplete_mask = (
        analysis_df[label_columns]
        .eq("")
        .any(axis=1)
    )

    incomplete = analysis_df.loc[
        incomplete_mask,
        "problem_id",
    ].tolist()

    if incomplete:
        raise ValueError(
            "Unlabeled or partially labeled problems: "
            + ", ".join(incomplete)
        )

    print("[PASS] All cases are fully labeled.")
    
validate_failure_labels(analysis_df)

[PASS] All cases are fully labeled.


In [16]:
tables = {
    "Plan Correctness": (
        analysis_df["plan_correctness"]
        .value_counts(dropna=False)
    ),

    "Implementation Fidelity": (
        analysis_df["implementation_fidelity"]
        .value_counts(dropna=False)
    ),

    "Primary Bottleneck": (
        analysis_df["primary_bottleneck"]
        .value_counts(dropna=False)
    ),

    "Plan × Implementation": (
        pd.crosstab(
            analysis_df["plan_correctness"],
            analysis_df["implementation_fidelity"],
            margins=True,
        )
    ),

    "Plan × Final Status": (
        pd.crosstab(
            analysis_df["plan_correctness"],
            analysis_df["status"],
            margins=True,
        )
    ),
}

for title, table in tables.items():
    print("=" * 80)
    print(title)
    print("=" * 80)
    display(table)

Plan Correctness


plan_correctness
Incorrect            6
Partially Correct    3
Correct              1
Name: count, dtype: int64

Implementation Fidelity


implementation_fidelity
Medium    5
High      3
Low       2
Name: count, dtype: int64

Primary Bottleneck


primary_bottleneck
Both        5
Planning    3
Coding      2
Name: count, dtype: int64

Plan × Implementation


implementation_fidelity,High,Low,Medium,All
plan_correctness,,,,
Correct,0,0,1,1
Incorrect,3,1,2,6
Partially Correct,0,1,2,3
All,3,2,5,10


Plan × Final Status


status,RUNTIME_ERROR,WRONG_ANSWER,All
plan_correctness,,,
Correct,0,1,1
Incorrect,1,5,6
Partially Correct,0,3,3
All,1,9,10


In [17]:
# def summarize(series):
#     return pd.DataFrame({
#         "count": series.value_counts(dropna=False),
#         "ratio": (
#             series.value_counts(normalize=True, dropna=False) * 100
#         ).round(1)
#     })

# tables = {
#     "Plan Correctness":
#         summarize(analysis_df["plan_correctness"]),

#     "Implementation Fidelity":
#         summarize(analysis_df["implementation_fidelity"]),

#     "Primary Bottleneck":
#         summarize(analysis_df["primary_bottleneck"]),

#     "Plan × Implementation":
#         pd.crosstab(
#             analysis_df["plan_correctness"],
#             analysis_df["implementation_fidelity"],
#             margins=True,
#         ),

#     "Plan × Final Status":
#         pd.crosstab(
#             analysis_df["plan_correctness"],
#             analysis_df["status"],
#             margins=True,
#         ),
# }

# for title, table in tables.items():
#     print(f"\n{'='*80}")
#     print(title)
#     print('='*80)
#     display(table)

In [18]:
problem_ids = analysis_df["problem_id"].tolist()

for problem_id in problem_ids:
    print(problem_id)

1873_A
1873_B
1873_D
1883_B
1883_C
1899_A
1899_B
1899_C
1899_D
abc301_a


---


In [19]:
outcome_map = {
    "PASS": "PASS",
    "WRONG_ANSWER": "WA",
    "RUNTIME_ERROR": "RE",
    "TIMEOUT": "TLE",
    "SYNTAX_ERROR": "SE",
    "EXTRACTION_ERROR": "EE",
    "EVALUATION_ERROR": "EVAL_ERROR",
}

failure_summary = (
    analysis_df[
        [
            "problem_id",
            "plan_correctness",
            "implementation_fidelity",
            "status",
            "primary_bottleneck",
        ]
    ]
    .rename(
        columns={
            "problem_id": "Problem",
            "plan_correctness": "Plan",
            "implementation_fidelity": "Fidelity",
            "status": "Outcome",
            "primary_bottleneck": "Cause",
        }
    )
)

failure_summary["Outcome"] = (
    failure_summary["Outcome"]
    .map(outcome_map)
    .fillna(failure_summary["Outcome"])
)

display(failure_summary)

,Problem,Plan,Fidelity,Outcome,Cause
0,1873_A,Incorrect,High,WA,Planning
1,1873_B,Partially Correct,Low,WA,Coding
2,1873_D,Incorrect,Medium,WA,Both
3,1883_B,Partially Correct,Medium,WA,Both
4,1883_C,Incorrect,Low,WA,Both
5,1899_A,Incorrect,High,WA,Planning
6,1899_B,Incorrect,Medium,RE,Both
7,1899_C,Partially Correct,Medium,WA,Both
8,1899_D,Incorrect,High,WA,Planning
9,abc301_a,Correct,Medium,WA,Coding


In [ ]:
# Incorrect plan: 6/10
# Partially correct plan: 3/10
# Correct plan: 1/10
# Planning bottleneck: 3/10
# Coding bottleneck: 2/10
# Both: 5/10